In [78]:
import pandas as pd
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import pickle

In [79]:
ratings = pd.read_csv(
    "MovieLens 1M/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names = ['UserID','MovieID','Rating','Timestamp']
)



model = tf.keras.models.load_model('recommender_model_matrix_factorizartion_v5.keras')

user_emb   = model.get_layer('user_emb')
movie_emb  = model.get_layer('movie_emb')
user_bias  = model.get_layer('user_bias')
movie_bias = model.get_layer('movie_bias')

user_emb_matrix   = user_emb.get_weights()[0]
movie_emb_matrix  = movie_emb.get_weights()[0]
user_bias_matrix  = user_bias.get_weights()[0]
movie_bias_matrix = movie_bias.get_weights()[0]


global_mean = np.load('global_mean.npy')
global_std = np.load('global_std.npy')

with open('user_id_map.pkl', 'rb') as file:
    user_id_map = pickle.load(file)
    file.close()

with open('movie_id_map.pkl', 'rb') as file:
    movie_id_map = pickle.load(file)
    file.close()
    
with open('held_out_users.pkl', 'rb') as file:
    held_out_users = pickle.load(file)
    file.close()

with open('held_out_movies.pkl', 'rb') as file:
    held_out_movies = pickle.load(file)
    file.close()


with open('base_ratings.pkl', 'rb') as file:
    base_ratings = pickle.load(file)
    file.close()



In [80]:
new_ratings_df = ratings[ratings['MovieID'].isin(held_out_movies) | ratings['UserID'].isin(held_out_users)].copy()

In [81]:
print("User Emb Size : ",len(user_emb_matrix))

print("User id map max value : ",len(user_id_map.values()))

print("Movie Emb Size : ",len(movie_emb_matrix))

print("Movie id map max value : ",len(movie_id_map.values()))

User Emb Size :  5940
User id map max value :  5940
Movie Emb Size :  3606
Movie id map max value :  3606


In [82]:
new_user_idx = max(user_id_map.values()) + 1 
new_movie_idx = max(movie_id_map.values()) + 1 

for uid in new_ratings_df['UserID'].unique():
    if uid not in user_id_map:
        
        user_id_map[uid]    =  new_user_idx
        new_user_idx        += 1      
        
for mid in new_ratings_df['MovieID'].unique():
    if mid not in movie_id_map:
        
        movie_id_map[mid]    =  new_movie_idx
        new_movie_idx        += 1      

In [83]:
print("User Emb Size : ",len(user_emb_matrix))

print("User id map max value : ",len(user_id_map.values()))

print("Movie Emb Size : ",len(movie_emb_matrix))

print("Movie id map max value : ",len(movie_id_map.values()))

User Emb Size :  5940
User id map max value :  6040
Movie Emb Size :  3606
Movie id map max value :  3706


both have 100 new users and movies, which we kept aside before training for simulation addition of new data into the model

In [ ]:

def add_new_users_and_movies(model, old_df, new_ratings_df, user_id_map, movie_id_map, global_mean, global_std):

    # next id for id map
    new_user_idx = max(user_id_map.values()) + 1 
    new_movie_idx = max(movie_id_map.values()) + 1 

    for uid in new_ratings_df['UserID'].unique():
        if uid not in user_id_map:
            
            user_id_map[uid]    =  new_user_idx
            new_user_idx        += 1      
            
    for mid in new_ratings_df['MovieID'].unique():
        if mid not in movie_id_map:
            
            movie_id_map[mid]    =  new_movie_idx
            new_movie_idx        += 1      


    # get new sizes
    
    new_num_users = max(user_id_map.values()) + 1
    new_num_movies = max(movie_id_map.values()) + 1
    
    # get old weights
    old_user_emb    = model.get_layer("user_emb").get_weights()[0]
    old_movie_emb   = model.get_layer("movie_emb").get_weights()[0]
    old_user_bias   = model.get_layer("user_bias").get_weights()[0]
    old_movie_bias  = model.get_layer("movie_bias").get_weights()[0]
    num_factors     = old_user_emb.shape[1]

    # expand with new random rows
    def expand(old_weights, new_size):
        extra = new_size - len(old_weights)
        
        if extra > 0:
            new_rows = np.random.normal(0,0.01,(extra, old_weights.shape[1]))
            return np.vstack([old_weights, new_rows])
        
        return old_weights   
    
    
    
    # prepare data

    old_sample_size = min(len(old_df), 3 * len(new_ratings_df))
    old_df = old_df.sample(old_sample_size, random_state=42).copy()
    
    new_ratings_df = new_ratings_df.copy()
    new_ratings_df['user_idx'] = new_ratings_df['UserID'].map(user_id_map)
    new_ratings_df['movie_idx'] = new_ratings_df['MovieID'].map(movie_id_map)
    
    combined_ratings = pd.concat([old_df, new_ratings_df])
    
    new_user_arr    = np.array(combined_ratings['user_idx']) 
    new_movie_arr   = np.array(combined_ratings['movie_idx'])
    
    new_rating_arr  = (np.array(combined_ratings['Rating']) - global_mean) / global_std
    
    
    
    # shuffle
    indices = np.random.permutation(len(new_user_arr))
    new_user_arr = new_user_arr[indices]
    new_movie_arr = new_movie_arr[indices]
    new_rating_arr = new_rating_arr[indices]
    
    
    # New Embeddings
    new_user_emb = expand(old_user_emb, new_num_users)
    new_movie_emb = expand(old_movie_emb, new_num_movies)
    new_user_bias = expand(old_user_bias, new_num_users)
    new_movie_bias = expand(old_movie_bias, new_num_movies)
    
    
    
    
    
    
    # rebuild model with exact architecture
    user_input  = keras.Input(shape = (1,))
    movie_input = keras.Input(shape = (1,))
    
    # creating embeddings
    user_vec = keras.layers.Embedding(
        input_dim=new_num_users,
        output_dim=num_factors,
        embeddings_regularizer=keras.regularizers.l2(1e-5),
        embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
        name="user_emb"
    )(user_input)
    
    movie_vec = keras.layers.Embedding(
        input_dim=new_num_movies,
        output_dim=num_factors,
        embeddings_regularizer=keras.regularizers.l2(1e-5),
        embeddings_initializer=keras.initializers.RandomNormal(mean=0, stddev=0.1),
        name="movie_emb"
    )(movie_input)
    
    user_bias   = keras.layers.Embedding(new_num_users, 1, name = 'user_bias')(user_input)
    movie_bias   = keras.layers.Embedding(new_num_movies, 1, name = 'movie_bias')(movie_input)
    
    user_vec    = keras.layers.Flatten()(user_vec)
    movie_vec   = keras.layers.Flatten()(movie_vec)
    user_bias   = keras.layers.Flatten()(user_bias)
    movie_bias  = keras.layers.Flatten()(movie_bias)
    
    dot         = keras.layers.Dot(axes=1)([user_vec,movie_vec])
    output      = keras.layers.Add()([dot, user_bias, movie_bias])
    
    new_model = keras.Model([user_input, movie_input], output)
    
    # load the expnded weights into the model
    new_model.get_layer('user_emb').set_weights([new_user_emb])
    new_model.get_layer('movie_emb').set_weights([new_movie_emb])
    new_model.get_layer('user_bias').set_weights([new_user_bias])
    new_model.get_layer('movie_bias').set_weights([new_movie_bias])
    
    
    new_model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=0.0001),
        loss = 'mse',
        metrics=[keras.metrics.RootMeanSquaredError(),'mae']
    )
    

    
    early_stopping = keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience=15,
        restore_best_weights= True
    )
    
    new_model.fit(
        [new_user_arr,new_movie_arr],
        new_rating_arr,
        epochs=500,
        batch_size=256,
        validation_split = 0.3,
        callbacks = [early_stopping],
        verbose = 1
    )
    
    return new_model, user_id_map, movie_id_map

Discarded Code:

* Cannot resize embedding layers after model is built.
* Keras requires weight shapes to remain fixed, so expanding input_dim (adding new users/movies) causes a shape mismatch in set_weights().

In [84]:

# def add_new_users_and_movies(model, new_ratings_df, user_id_map, movie_id_map, global_mean, global_std):

#     # next id for id map
#     new_user_idx = max(user_id_map.values()) + 1 
#     new_movie_idx = max(movie_id_map.values()) + 1 

#     for uid in new_ratings_df['UserID'].unique():
#         if uid not in user_id_map:
            
#             user_id_map[uid]    =  new_user_idx
#             new_user_idx        += 1      
            
#     for mid in new_ratings_df['MovieID'].unique():
#         if mid not in movie_id_map:
            
#             movie_id_map[mid]    =  new_movie_idx
#             new_movie_idx        += 1      

#     for layer_name, id_map in [
#             ('user_emb', user_id_map),
#             ('movie_emb', movie_id_map),
#             ('user_bias', user_id_map),
#             ('movie_bias', movie_id_map),
#             ]:
     
#         layer = model.get_layer(layer_name)
#         old_weights = layer.get_weights()[0]
        
#         # doing plus one, as id map values start with 0, so if max = 10, then 10 + 0th index = 11
#         new_size = max(id_map.values()) + 1  
        
#         if new_size > len(old_weights):
#             extra_rows  = new_size - len(old_weights)
            
#             new_rows    =  np.random.normal(0,0.01, (extra_rows, old_weights.shape[1]))
            
#             new_weights = np.vstack([old_weights, new_rows]) # appends the arrays
            
#             layer.set_weights([new_weights])


        
#     for layer in model.layers:
#         layer.trainable = False
        
#     model.get_layer("user_emb").trainable = True
#     model.get_layer("movie_emb").trainable = True
#     model.get_layer("user_bias").trainable = True
#     model.get_layer("movie_bias").trainable = True

    

#     # get idx values pointing the user/movie id to their embedding index
#     new_ratings_df['user_idx'] = new_ratings_df['UserID'].map(user_id_map)
#     new_ratings_df['movie_idx'] = new_ratings_df['MovieID'].map(movie_id_map)
    
#     new_user_arr    = np.array(new_ratings_df['user_idx'])
#     new_movie_arr   = np.array(new_ratings_df['movie_idx'])
    
#     new_rating_arr  = (np.array(new_ratings_df['Rating']) - global_mean) / global_std
    
    
#     #Training the model on new data
    
#     # shuffle
#     indices        = np.random.permutation(len(new_user_arr))
#     new_user_arr   = new_user_arr[indices]
#     new_movie_arr  = new_movie_arr[indices]
#     new_rating_arr = new_rating_arr[indices]
    
#     model.compile(
#         optimizer = keras.optimizers.Adam(learning_rate= 0.0001),
#         loss = 'mse',
#         metrics = ['mae',keras.metrics.RootMeanSquaredError()]
#     )
       
#     early_stopping = keras.callbacks.EarlyStopping(
#     monitor='val_loss',
#     patience=15,        # stop if no improvement for 5 epochs
#     restore_best_weights=True  # keep the best model
# )

#     # train 
#     model.fit(
#         [new_user_arr, new_movie_arr],    
#         new_rating_arr,
#         epochs = 150,
#         batch_size = 512,
#         validation_split = 0.3,
#         callbacks=[early_stopping],
#         verbose = 1
#     ) 
    
    
#     return model, user_id_map, movie_id_map
    

In [86]:
new_ratings_df.shape

(29932, 4)

In [88]:
model, user_id_map, movie_id_map = add_new_users_and_movies(model, base_ratings, new_ratings_df, user_id_map, movie_id_map, global_mean, global_std)

Epoch 1/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.7084 - mae: 0.6267 - root_mean_squared_error: 0.7981 - val_loss: 0.7045 - val_mae: 0.6254 - val_root_mean_squared_error: 0.7964
Epoch 2/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6984 - mae: 0.6222 - root_mean_squared_error: 0.7930 - val_loss: 0.7009 - val_mae: 0.6244 - val_root_mean_squared_error: 0.7950
Epoch 3/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6891 - mae: 0.6177 - root_mean_squared_error: 0.7879 - val_loss: 0.6977 - val_mae: 0.6234 - val_root_mean_squared_error: 0.7936
Epoch 4/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6803 - mae: 0.6134 - root_mean_squared_error: 0.7829 - val_loss: 0.6949 - val_mae: 0.6225 - val_root_mean_squared_error: 0.7924
Epoch 5/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6719 - mae: 0.6092 - root_mean_squared_error: 0.7780 - val_loss: 0.6924 - val_mae: 0.6216 - val_root_mean_squared_error: 0.7912
Epoch 6/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 3

In [ ]:
idx2movie = {v:k for k,v in movie_id_map.items()}

In [6]:
## Saving the id mapping
import pickle
model.save("recommender_model_cf_final.keras")

with open("full_user_id_map.pkl", "wb") as f:
    pickle.dump(user_id_map, f)

with open("full_movie_id_map.pkl", "wb") as f:
    pickle.dump(movie_id_map, f)

with open("idx2movie.pkl", "wb") as f:
    pickle.dump(idx2movie, f)


with open("base_ratings.pkl", "wb") as f:
    pickle.dump(base_ratings, f)





# global mean
np.save("global_mean.npy", global_mean)

np.save("global_std.npy", global_std)
    